# Setup check

Run this notebook once before Day 1. It imports every library the labs use, reads one data file, asks
each hosted model one question (or says that the scripted stand-in will answer), and loads the course's
own small model. Every line that prints "missing" tells you what to install or set.

In [ ]:
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/ae-310").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/SDAIAAcademy/ae-310.git", "/content/ae-310"], check=True)
    os.chdir("/content/ae-310/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    try:  # the key from the Secrets panel, when it is there
        from google.colab import userdata
        os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY") or "")
    except Exception:
        pass
else:
    while not Path("qayd.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import chamber as C
import qayd as Q
Q.banner("Setup check")

In [ ]:
from importlib import metadata

# Versions are read from the installed packages without importing them: Locust, for one, changes
# how the network works when it is imported, which a notebook must not do to itself.
missing = []
for name in ("numpy", "matplotlib", "torch", "transformers", "peft", "tokenizers", "openai", "httpx", "fastapi", "uvicorn", "locust"):
    try:
        print(f"ok       {name} {metadata.version(name)}")
    except metadata.PackageNotFoundError:
        missing.append(name)
        print(f"missing  {name}")
print()
print("data:", len(Q.load("requests_100")), "requests in data/requests_100.jsonl")

In [ ]:
import prompts

for role in ("small", "large"):
    reply = Q.chat([{"role": "system", "content": prompts.FROZEN}, {"role": "user", "content": Q.load("requests_100")[0]["text"]}], role)
    print(f"{role:5s} model answered from: {reply['source']:8s} {reply['text'][:80]!r}")
if Q.using_standin():
    print("\nNo working key: the scripted stand-in answers in Days 1 and 2. To use the hosted models, put OPENROUTER_API_KEY")
    print("in labs/.env (locally) or in the Secrets panel (Colab) and run this notebook again.")

In [ ]:
import torch

import tuning as U

print("tier:", U.tier(), "· device:", U.device())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, {gpu.total_memory / 2**30:.1f} GB")
else:
    print("No GPU: Days 3 and 4 train the course's own small model on this processor. That is a supported way to take the course.")
model, tok = U.load_base("tiny")
answer = U.generate(model, tok, ["Translate to Arabic: Riyadh"], which="tiny")[0]
print("course model:", U.count_parameters(model)["total"], "parameters; it answers:", answer)
assert answer.strip() == "الرياض", "the course model did not load correctly"
print()
print("All set." if not missing else f"Install what is missing: {missing}")